# Engenharia de Contexto para RAG sobre Dados Tabulares — FinQA

Estudo controlado de como a **representação da tabela no prompt** afeta a **acurácia** e o **custo em tokens** de LLMs abertos de ~8B (Llama 3.1 8B, Qwen3-8B, Phi-4-mini) no FinQA, sem ajuste fino.

**Representações** (texto do relatório fixo em todas):
1. **Markdown** — tabela completa
2. **JSON** — tabela completa (lista de objetos por linha)
3. **Linearizada** — cada célula vira uma frase com cabeçalhos de linha e coluna
4. **Seletiva** — cabeçalho + linhas relevantes à pergunta (sem gabarito), em Markdown
5. *Diagnóstico:* **Gold** — cabeçalho + linhas do gabarito (teto da seleção)

**Análise** por número de passos do programa, origem da evidência (tabela/texto/ambos) e tamanho da tabela.

Repositório: https://github.com/Arthur-Marques-IA/NLP_Fin-RAG

In [ ]:
# Ambiente: Python, PyTorch e GPU
import sys, os, torch
print("Python", sys.version.split()[0])
print("torch", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print("GPU:", p.name, f"{p.total_memory/1e9:.1f} GB")
else:
    print("ATENÇÃO: sem GPU — troque o runtime para GPU antes da etapa de inferência.")

## 1. Dados — FinQA (conjunto de teste oficial)

Baixamos o `test.json` do repositório oficial ([czyssrs/FinQA](https://github.com/czyssrs/FinQA)). Cada exemplo traz `pre_text`, `table`, `post_text` e o bloco `qa` com pergunta, programa anotado (`program`), resposta executada (`exe_ans`) e evidências do gabarito (`gold_inds`).

In [ ]:
# Download do FinQA (teste)
import json, pathlib, urllib.request

DATA_DIR = pathlib.Path("data")
DATA_DIR.mkdir(exist_ok=True)
FINQA_URL = "https://raw.githubusercontent.com/czyssrs/FinQA/main/dataset/test.json"
test_path = DATA_DIR / "finqa_test.json"
if not test_path.exists():
    urllib.request.urlretrieve(FINQA_URL, test_path)

with open(test_path, encoding="utf-8") as f:
    finqa_test = json.load(f)
print("Perguntas no teste:", len(finqa_test))
print("Chaves de um exemplo:", list(finqa_test[0].keys()))
print("Chaves de qa:", list(finqa_test[0]["qa"].keys()))

In [ ]:
# Inspeção de um exemplo
ex = finqa_test[0]
qa = ex["qa"]
print("ID:", ex["id"])
print("Pergunta:", qa["question"])
print("Programa:", qa["program"])
print("Resposta (exe_ans):", qa["exe_ans"], "| answer:", qa["answer"])
print("Linhas de tabela no gabarito (ann_table_rows):", qa["ann_table_rows"])
print("Linhas de texto no gabarito (ann_text_rows):", qa["ann_text_rows"])
print("gold_inds:", qa["gold_inds"])
print(f"\nTabela ({len(ex['table'])} linhas):")
for row in ex["table"]:
    print("  ", row)
print(f"\npre_text: {len(ex['pre_text'])} frases | post_text: {len(ex['post_text'])} frases")

## 2. Metadados para a análise

Para cada pergunta extraímos as variáveis de corte do estudo:
- **`n_steps`** — número de operações no programa anotado (1, 2, 3+), proxy de complexidade do raciocínio;
- **`evidence`** — origem da evidência no gabarito: `table`, `text` ou `both` (a partir das chaves de `gold_inds`);
- **`n_rows` / `n_cols`** — tamanho da tabela (sem contar o cabeçalho).

> Os campos `table_retrieved*`, `text_retrieved*`, `tfidftopn` e `model_input` vêm de recuperadores do artigo original e **não** são usados nos prompts.

In [ ]:
# Metadados por pergunta
import re
import pandas as pd

OP_RE = re.compile(r"[a-z_]+\(")

def count_steps(program: str) -> int:
    return len(OP_RE.findall(program))

def evidence_source(gold_inds: dict) -> str:
    kinds = {k.split("_")[0] for k in gold_inds}
    if kinds == {"table"}:
        return "table"
    if kinds == {"text"}:
        return "text"
    return "both"

meta = pd.DataFrame([{
    "id": ex["id"],
    "n_steps": count_steps(ex["qa"]["program"]),
    "evidence": evidence_source(ex["qa"]["gold_inds"]),
    "n_rows": len(ex["table"]) - 1,
    "n_cols": max(len(r) for r in ex["table"]),
    "gold_answer": ex["qa"]["exe_ans"],
} for ex in finqa_test])
meta["steps_bin"] = meta["n_steps"].clip(upper=3).map({1: "1", 2: "2", 3: "3+"})

print("Passos:\n", meta["steps_bin"].value_counts().sort_index().to_string(), "\n")
print("Origem da evidência:\n", meta["evidence"].value_counts().to_string(), "\n")
print("Linhas de dados por tabela:\n", meta["n_rows"].describe().round(2).to_string())
print("\nTipos de resposta:", meta["gold_answer"].map(lambda v: type(v).__name__).value_counts().to_dict())

## 3. Representações da tabela

Todas partem da mesma `table` do FinQA (linha 0 = cabeçalho, coluna 0 = rótulo da linha) e mantêm os valores das células **sem alteração**, para que só a forma mude.

- **Seletiva:** pontua cada linha pela sobreposição lexical com a pergunta (termos sem stopwords, com peso extra para anos/números) e mantém as `k` melhores, na ordem original. Não usa o gabarito nem LLM, então não consome tokens de modelo — o custo da seleção é só de CPU.
- **Gold:** cabeçalho + linhas do gabarito, tiradas das chaves `table_i` de `gold_inds` (o campo `ann_table_rows` vem vazio em ~110 perguntas) — diagnóstico do teto da seleção.

In [ ]:
# Serializações da tabela
def _header_names(header):
    """Nomes de coluna únicos; a primeira coluna vazia vira 'item'."""
    names, seen = [], {}
    for i, h in enumerate(header):
        name = h.strip() or ("item" if i == 0 else f"col_{i}")
        if name in seen:
            seen[name] += 1
            name = f"{name}_{seen[name]}"
        else:
            seen[name] = 0
        names.append(name)
    return names

def _pad(row, n):
    return list(row) + [""] * (n - len(row))

def to_markdown(table):
    if not table:
        return ""
    n = max(len(r) for r in table)
    header = _header_names(_pad(table[0], n))
    esc = lambda c: str(c).replace("|", "\\|").strip()
    lines = ["| " + " | ".join(map(esc, header)) + " |",
             "|" + "|".join(["---"] * n) + "|"]
    lines += ["| " + " | ".join(map(esc, _pad(r, n))) + " |" for r in table[1:]]
    return "\n".join(lines)

def to_json(table):
    if not table:
        return "[]"
    n = max(len(r) for r in table)
    header = _header_names(_pad(table[0], n))
    rows = [dict(zip(header, (str(c).strip() for c in _pad(r, n)))) for r in table[1:]]
    return json.dumps(rows, ensure_ascii=False)

def to_linearized(table):
    """Uma frase por célula: 'the <coluna> of <linha> is <valor> ;' (template do FinQA)."""
    if not table:
        return ""
    n = max(len(r) for r in table)
    header = _pad(table[0], n)
    sents = []
    for r in table[1:]:
        r = _pad(r, n)
        label = r[0].strip()
        for col, val in zip(header[1:], r[1:]):
            if str(val).strip():
                sents.append(f"the {col.strip()} of {label} is {str(val).strip()} ;")
    return "\n".join(sents)

# --- Gabarito de linhas ---
def gold_table_rows(ex):
    """Linhas de tabela do gabarito, a partir das chaves 'table_i' de gold_inds.
    (ann_table_rows vem vazio em ~110 perguntas do teste; gold_inds é mais completo.)"""
    return sorted(int(k.split("_")[1]) for k in ex["qa"]["gold_inds"] if k.startswith("table_"))

# --- Seleção de linhas (sem gabarito) ---
STOPWORDS = set("""a an the of in on for to by and or is are was were be been what which how
much many did does do from with as at that this these those its it their than then
percent percentage change total amount value values ratio year years""".split())
TOKEN_RE = re.compile(r"[a-z]+|\d+(?:\.\d+)?")

def _terms(text):
    return [t for t in TOKEN_RE.findall(str(text).lower()) if t not in STOPWORDS]

def select_rows(table, question, k=3):
    """Índices (>=1) das k linhas com maior sobreposição com a pergunta, na ordem original."""
    q = set(_terms(question))
    scored = []
    for i, row in enumerate(table[1:], start=1):
        terms = set(_terms(" ".join(map(str, row))))
        score = sum(2.0 if t.isdigit() else 1.0 for t in q & terms)
        scored.append((score, i))
    top = sorted(scored, key=lambda s: (-s[0], s[1]))[:k]
    return sorted(i for _, i in top)

def subtable(table, row_ids):
    """Cabeçalho (linha 0) sempre incluído + linhas pedidas."""
    return [table[0]] + [table[i] for i in row_ids if 0 < i < len(table)]

def to_selective(table, question, k=3):
    return to_markdown(subtable(table, select_rows(table, question, k)))

REPRESENTATIONS = ["markdown", "json", "linearized", "selective", "gold"]

def serialize(ex, rep, k=3):
    t, qa = ex["table"], ex["qa"]
    if rep == "markdown":   return to_markdown(t)
    if rep == "json":       return to_json(t)
    if rep == "linearized": return to_linearized(t)
    if rep == "selective":  return to_selective(t, qa["question"], k)
    if rep == "gold":       return to_markdown(subtable(t, gold_table_rows(ex)))
    raise ValueError(rep)

def build_context(ex, rep, k=3):
    """Texto do relatório fixo; só a tabela muda entre representações."""
    pre = " ".join(ex["pre_text"])
    post = " ".join(ex["post_text"])
    return f"{pre}\n\nTable:\n{serialize(ex, rep, k)}\n\n{post}".strip()

for rep in REPRESENTATIONS:
    print(f"===== {rep} =====")
    print(serialize(finqa_test[0], rep), "\n")

In [ ]:
# Cobertura da seleção: a seletiva preserva as linhas do gabarito? (H2)
def selection_recall(ex, k):
    gold = {i for i in gold_table_rows(ex) if i > 0}  # linha 0 (cabeçalho) sempre entra
    if not gold:
        return None  # sem linha de dados no gabarito: nada a preservar
    return len(gold & set(select_rows(ex["table"], ex["qa"]["question"], k))) / len(gold)

n_with_rows = sum(1 for ex in finqa_test if any(i > 0 for i in gold_table_rows(ex)))
print(f"Perguntas com linhas de dados no gabarito: {n_with_rows} de {len(finqa_test)}\n")
for k in (1, 2, 3, 4, 5):
    rec = pd.Series([selection_recall(ex, k) for ex in finqa_test], dtype="float").dropna()
    print(f"k={k}: recall médio {rec.mean():.3f} | todas as linhas gold preservadas em {(rec == 1).mean():.1%}")

meta["sel_recall_k3"] = [selection_recall(ex, 3) for ex in finqa_test]
print("\nRecall (k=3) por número de passos:")
print(meta.groupby("steps_bin")["sel_recall_k3"].mean().round(3).to_string())

## 4. Avaliação — acurácia numérica com tolerância

O gabarito é `exe_ans` (resultado da execução do programa anotado), como na acurácia de execução do FinQA. Extraímos o número da resposta do modelo (o primeiro após `Answer:`, ou o último do texto), tratando `$`, `,`, `%`, `.5` e negativos entre parênteses. A resposta é **correta** se, em alguma escala (×1, ×100, ÷100 — o FinQA mistura razões e percentuais):
- o **erro relativo é ≤ 1%**; ou
- vale a **tolerância de arredondamento**: o gabarito arredondado com as casas decimais que o modelo usou é igual à resposta (`7%` ↔ `0.07157`), com o mesmo sinal e resposta ≠ 0.

Perguntas **sim/não** (20) são comparadas como string.

Validação: o campo `answer` escrito pelos anotadores é aceito em 91,6% dos casos; o resto é ruído de anotação (campo vazio, texto, sinal ou escala trocados), não falha do parser.

In [ ]:
# Extração e comparação de respostas
import math

NUM_RE = re.compile(r"\(?-?\$?\s*(?:\d[\d,]*(?:\.\d+)?|\.\d+)\s*%?\)?")

def _parse_token(text):
    """(valor, casas_decimais) do número relevante de `text`: o primeiro após 'Answer:',
    ou o último do texto se não houver 'Answer:'. None se não houver número."""
    if text is None:
        return None
    s = str(text)
    m = re.search(r"answer\s*[:：]\s*(.*)", s, flags=re.I | re.S)
    if m:
        s = m.group(1)
    cands = NUM_RE.findall(s)
    if not cands:
        return None
    tok = (cands[0] if m else cands[-1]).strip()
    neg = (tok.startswith("(") and tok.endswith(")")) or tok.lstrip("(").startswith("-")
    digits = re.sub(r"[^\d.]", "", tok)
    decimals = len(digits.split(".")[1]) if "." in digits else 0
    val = float(digits)
    return (-val if neg else val), decimals

def parse_number(text):
    p = _parse_token(text)
    return None if p is None else p[0]

def parse_yes_no(text):
    m = re.search(r"\b(yes|no)\b", str(text).lower())
    return m.group(1) if m else None

def is_correct(pred_text, gold, rel_tol=0.01):
    """Correta se, para alguma escala (x1, /100, x100 — o FinQA mistura razão e %):
    (a) erro relativo <= rel_tol; ou
    (b) tolerância de arredondamento: o gabarito arredondado com as casas decimais
        que a resposta usou é igual à resposta (mesmo sinal, resposta != 0)."""
    if isinstance(gold, str):  # 'yes' / 'no'
        return parse_yes_no(pred_text) == gold.strip().lower()
    p = _parse_token(pred_text)
    if p is None or gold is None or (isinstance(gold, float) and math.isnan(gold)):
        return False
    pred, dec = p
    if gold == 0:
        return abs(pred) < 1e-6
    for g in (gold, gold * 100, gold / 100):  # gabarito na escala da resposta
        if math.isclose(pred, g, rel_tol=rel_tol):
            return True
        if pred != 0 and (pred > 0) == (g > 0) and math.isclose(round(g, dec), pred, abs_tol=1e-9):
            return True
    return False

# Testes unitários
tests = [
    ("Answer: 94", 94.0, True),
    ("The change is $ 94 million.\nAnswer: 94.0", 94.0, True),
    ("Answer: 12.3%", 0.12298, True),
    ("Answer: 12.35", 12.345678, True),
    ("Answer: 7%", 0.07157, True),        # arredondado para inteiro
    ("Answer: 14%", 0.14464, True),
    ("Answer: .385", 0.38537, True),      # sem zero à esquerda
    ("Answer: (32)", -32.0, True),
    ("Answer: -32", -32.0, True),
    ("Answer: 1,234.5", 1234.5, True),
    ("Answer: 95", 94.0, False),
    ("Answer: 0.004", 0.0012, False),
    ("Answer: 0", 0.4, False),            # zero não ganha por arredondamento
    ("Answer: 0.35%", -0.00357, False),   # sinal trocado
    ("Answer: yes", "yes", True),
    ("I cannot determine this.", 94.0, False),
]
for text, gold, expected in tests:
    got = is_correct(text, gold)
    print(("OK  " if got == expected else "ERRO"), repr(text)[:45].ljust(47), "gold", gold, "->", got)
assert all(is_correct(t, g) == e for t, g, e in tests)

# Validação: o campo 'answer' (texto do anotador) deve bater com exe_ans
ok = [is_correct(ex["qa"]["answer"], ex["qa"]["exe_ans"]) for ex in finqa_test]
print(f"\n'answer' do FinQA reconhecido como correto em {sum(ok)}/{len(ok)} ({sum(ok)/len(ok):.1%})")
print("Divergências restantes (ruído de anotação — escala, sinal ou valor diferente de exe_ans):")
shown = 0
for ex, good in zip(finqa_test, ok):
    if not good and shown < 6:
        print(f"   answer={ex['qa']['answer']!r:<12} exe_ans={ex['qa']['exe_ans']}")
        shown += 1

## 5. Custo — tokens de contexto por representação

Contamos os tokens do contexto (texto do relatório + tabela serializada) com o **tokenizador de cada modelo**. O Llama 3.1 é *gated* no Hugging Face: adicione um token com acesso ao modelo em **Secrets** do Colab com o nome `HF_TOKEN` (sem isso ele é pulado nesta etapa).

O custo da seleção (seletiva) é zero em tokens de LLM, porque a seleção é lexical.

In [ ]:
# Tokenizadores dos três modelos
from transformers import AutoTokenizer

MODELS = {
    "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
    "qwen3-8b":    "Qwen/Qwen3-8B",
    "phi4-mini":   "microsoft/Phi-4-mini-instruct",
}

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

tokenizers = {}
for name, repo in MODELS.items():
    try:
        tokenizers[name] = AutoTokenizer.from_pretrained(repo, token=HF_TOKEN)
        print(f"OK    {name:12s} ({repo})")
    except Exception as e:
        print(f"PULADO {name:12s} ({repo}): {type(e).__name__} — {str(e).splitlines()[0][:110]}")

In [ ]:
# Contagem de tokens: tabela isolada e contexto completo, por modelo e representação
records = []
for ex, m in zip(finqa_test, meta.itertuples()):
    for rep in REPRESENTATIONS:
        table_txt = serialize(ex, rep)
        ctx_txt = build_context(ex, rep)
        for name, tok in tokenizers.items():
            records.append({
                "id": ex["id"], "model": name, "rep": rep,
                "table_tokens": len(tok.encode(table_txt, add_special_tokens=False)),
                "context_tokens": len(tok.encode(ctx_txt, add_special_tokens=False)),
                "n_rows": m.n_rows, "steps_bin": m.steps_bin, "evidence": m.evidence,
            })
tokens_df = pd.DataFrame(records)
tokens_df.to_csv(DATA_DIR / "tokens_by_rep.csv", index=False)

summary = (tokens_df.groupby(["model", "rep"])[["table_tokens", "context_tokens"]]
           .mean().round(1).unstack("model"))
print("Média de tokens por pergunta:\n")
print(summary.reindex(REPRESENTATIONS).to_string())

base = tokens_df[tokens_df.rep == "markdown"].set_index(["id", "model"])["context_tokens"]
print("\nContexto relativo ao Markdown completo (média de razões):")
for rep in REPRESENTATIONS:
    r = tokens_df[tokens_df.rep == rep].set_index(["id", "model"])["context_tokens"] / base
    print(f"  {rep:11s} {r.mean():.3f}")

## Próximos passos

1. **Prompt fixo** — instrução + 2–3 exemplos (few-shot) retirados do **treino** do FinQA, pergunta sempre depois do contexto, saída terminando em `Answer: <número>`.
2. **Inferência** — Qwen3-8B, Phi-4-mini e Llama 3.1 8B (requer `HF_TOKEN`), temperatura 0, mesmas versões de modelo/tokenizador; na T4, os modelos de 8B em 4 bits (registrar como limitação).
3. **Resultados** — acurácia por modelo × representação, com IC por bootstrap pareado; cortes por `steps_bin`, `evidence` e `n_rows`; acurácia por mil tokens.
4. **Erros** — separar falha de extração e falha de cálculo (taxonomia de Srivastava et al.).